# Lab 4 — 서버 로그를 Poisson으로 적합하기

**확률통계 · Week 4 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 데이터의 **평균과 분산을 비교**해 분포를 짐작한다.
2. $\hat{\lambda} = \bar{X}$ 로 Poisson을 적합하고 히스토그램에 겹쳐 **눈으로 검증**한다.
3. **동질적이지 않은 구간**을 섞으면 적합이 무너지는 것을 확인한다.

⏱ **예상 소요 시간: 35분**

---

### 데이터

`data/w04_server_log.csv` — 하루치(1,440분) 서버 접속 로그.

| 열 | 뜻 |
|---|---|
| `minute` | 0~1439 (자정부터 몇 분) |
| `hour` | 0~23 |
| `requests` | **그 1분 동안 들어온 요청 수** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)

# Colab에서 파일을 업로드했다면 경로를 바꿔주면 된다
DATA = "w04_server_log.csv"

log = np.genfromtxt(DATA, delimiter=",", names=True, dtype=int)
hour, req = log["hour"], log["requests"]

print("총", len(req), "분 기록 / 총 요청", req.sum())
print("처음 10분:", req[:10])

## Part 1. 먼저 그려본다

분석의 첫 단계는 언제나 **그림**이다. 숫자만 보면 놓친다.

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(np.arange(len(req)) / 60, req, lw=0.8)
plt.xlabel("Hour of day")
plt.ylabel("Requests per minute")
plt.title("Server log")
plt.show()

낮에는 많고 새벽에는 적다. **시간대마다 성질이 다르다.**

그래서 우선 **성질이 비슷한 구간**만 떼어낸다. 10~15시를 쓰자.

### 실습 1 — 구간 잘라내고 요약하기

In [ ]:
sub = req[(hour >= 10) & (hour < 15)]

# TODO 1: 이 구간의 평균과 분산을 구하세요
mean_sub = 0.0
var_sub = 0.0

print(f"표본 수 : {len(sub)}")
print(f"평균    : {mean_sub:.3f}")
print(f"분산    : {var_sub:.3f}")

> **분산/평균 비율이 1에 가까운가?**
> Poisson의 가장 큰 특징이 $\mathbb{E}[X] = \mathrm{Var}[X] = \lambda$ 라는 것이었다.
> 이 비율이 1 근처면 Poisson을 후보로 올릴 만하다.

## Part 2. Poisson 적합

모수 $\lambda$ 를 정해야 한다. 무엇으로 정할까? — **표본평균**을 쓴다.

$$\hat{\lambda} = \bar{X}$$

> 왜 표본평균이 최선인지는 **12주차(MLE)** 에서 증명한다. 오늘은 "그렇게 한다"로 충분하다.

### 실습 2 — 적합하고 겹쳐 그리기

In [ ]:
lam_hat = sub.mean()
k = np.arange(sub.min(), sub.max() + 1)

# TODO 2: Poisson PMF를 계산하세요.  힌트: stats.poisson.pmf(k, lam_hat)
pmf = np.zeros(len(k))

plt.figure(figsize=(7, 4))
plt.hist(sub, bins=np.arange(sub.min() - 0.5, sub.max() + 1.5),
         density=True, alpha=0.8, label="observed log")
plt.plot(k, pmf, "o-", color="darkorange", label=f"Poisson(lambda={lam_hat:.2f})")
plt.xlabel("Requests per minute")
plt.ylabel("Relative frequency")
plt.title("10:00-15:00")
plt.legend()
plt.show()

## Part 3. 구간을 잘못 고르면

이번에는 **하루 전체**를 하나의 Poisson으로 보자. 새벽(평균 1건)과 낮(평균 8건)을 섞는 것이다.

### 실습 3 — 전체 데이터로 같은 적합

In [ ]:
lam_all = req.mean()
k_all = np.arange(req.min(), req.max() + 1)

plt.figure(figsize=(7, 4))
plt.hist(req, bins=np.arange(req.min() - 0.5, req.max() + 1.5),
         density=True, alpha=0.8, label="whole day")
plt.plot(k_all, stats.poisson.pmf(k_all, lam_all), "o-", color="darkorange",
         label=f"Poisson(lambda={lam_all:.2f})")
plt.xlabel("Requests per minute")
plt.ylabel("Relative frequency")
plt.title("Whole day - does it fit?")
plt.legend()
plt.show()

# TODO 3: 전체 데이터의 분산/평균 비율을 출력하세요
print("전체 분산/평균 비율:", None)

🤔 **적합이 무너졌다.** 관측 히스토그램이 Poisson보다 훨씬 넓게 퍼져 있다.
분산/평균 비율도 1을 크게 넘는다 — 이런 현상을 **overdispersion**이라고 한다.

> **원인은 $\lambda$ 가 하나가 아니기 때문이다.**
> 새벽의 $\lambda$ 와 낮의 $\lambda$ 가 다른데 억지로 하나로 뭉갠 것이다.
> **"어떤 구간을 하나의 모형으로 볼 것인가"** 가 모델링의 절반이다.

## Part 4. Binomial → Poisson 직접 확인

Poisson이 "드문 일을 아주 많이 시도한 결과"라는 것을 눈으로 확인하자.
$np = \lambda = 3$ 을 고정한 채 $n$ 을 키운다.

### 실습 4

In [ ]:
LAM = 3
k = np.arange(0, 13)

plt.figure(figsize=(8, 4))
for n in [10, 50, 500]:
    p = LAM / n
    # TODO 4: Binomial(n, p) 의 PMF를 그리세요.  힌트: stats.binom.pmf(k, n, p)
    pass

plt.plot(k, stats.poisson.pmf(k, LAM), "s-", lw=2.5, color="black",
         label="Poisson(3)")
plt.xlabel("x")
plt.ylabel("p(x)")
plt.title("np = 3 fixed, n increasing")
plt.legend()
plt.show()

## Part 5. 분포 고르기 — 직접 판단해보기

아래 세 데이터는 각각 다른 메커니즘으로 만들어졌다. 어떤 분포일까?

### 실습 5 — 평균과 분산으로 추리하기

In [ ]:
rng = np.random.default_rng(20260302)
mystery = {
    "A": rng.poisson(4, size=2000),
    "B": rng.binomial(20, 0.2, size=2000),
    "C": rng.geometric(0.2, size=2000),
}

for name, data in mystery.items():
    # TODO 5: 각 데이터의 평균, 분산, 분산/평균 비율을 출력하세요
    print(name, "평균/분산을 계산해보세요")

**판단 기준**

| 관찰 | 시사점 |
|---|---|
| 분산/평균 ≈ 1 | Poisson |
| 분산/평균 < 1 | Binomial (분산 $np(1-p)$ < 평균 $np$) |
| 분산/평균 > 1, 오른쪽 꼬리가 길다 | Geometric 계열 |

정답을 코드에서 확인할 수 있다(`rng.poisson`, `rng.binomial`, `rng.geometric`).
중요한 것은 **숫자만 보고 메커니즘을 추리하는 연습**이다.

---

## 마무리 — 자가 점검

- [ ] 평균과 분산의 비율로 분포를 짐작할 수 있다
- [ ] $\hat{\lambda} = \bar{X}$ 로 Poisson을 적합할 수 있다
- [ ] 동질적이지 않은 구간을 섞으면 적합이 무너진다는 것을 확인했다
- [ ] Binomial이 Poisson으로 수렴하는 것을 직접 그렸다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)